In [4]:
'''-- Выгружаем данные с уже посчитанными базовыми метриками для POWER BI
SELECT 
    "Расположение",
    "Услуга",
    "Вид запроса",
    "Приоритет",
    COUNT(*) as количество_обращений,
    AVG(EXTRACT(EPOCH FROM ("Фактическое время выполнения" - "Дата/время регистрации"))/3600) as среднее_время_решения_часы,
    SUM(CASE WHEN "Просрочен" = 'просрочен' THEN 1 ELSE 0 END) as просроченные
FROM support_tickets_final_anonymized
WHERE "Дата/время регистрации" >= '2025-05-18' -- last 6 month
GROUP BY 1,2,3,4
ORDER BY количество_обращений DESC;'''

'-- Выгружаем данные с уже посчитанными базовыми метриками\nSELECT \n    "Расположение",\n    "Услуга",\n    "Вид запроса",\n    "Приоритет",\n    COUNT(*) as количество_обращений,\n    AVG(EXTRACT(EPOCH FROM ("Фактическое время выполнения" - "Дата/время регистрации"))/3600) as среднее_время_решения_часы,\n    SUM(CASE WHEN "Просрочен" = \'просрочен\' THEN 1 ELSE 0 END) as просроченные\nFROM tickets\nWHERE "Дата/время регистрации" >= \'2020-07-01\'  -- фильтр за 6 месяцев\nGROUP BY 1,2,3,4\nORDER BY количество_обращений DESC;'

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

In [14]:
df = pd.read_csv(r'C:\Users\Incognitus\Documents\git_hub\Simulate\for_resume_project\support_tickets_final_anonymized.csv', parse_dates=['Дата/время регистрации', 'Фактическое время выполнения'])

In [16]:
# Проверка распределения по станциям до фильтра
print("Распределение по станциям (все данные):")
print(df['Расположение'].value_counts().head(15))

Распределение по станциям (все данные):
Расположение
АЭС-2 (Западная)                      14334
АЭС-3 (Восточная)                      3609
Промплощадка Альфа                      339
АЭС-6 (Прибрежная)                      218
АЭС-1 (Северная)                        165
АЭС-5 (Центральная)                     148
Хранилище отработанного топлива №2       50
АЭС-4 (Южная)                            45
Промплощадка Бета                        45
Лабораторный корпус                      19
Хранилище отработанного топлива №1        8
Резервный пункт управления                4
Административно-бытовой корпус            3
Учебно-тренировочный центр                1
Здание управления (ЗУ-1)                  1
Name: count, dtype: int64


In [9]:
df.head(5)

,Дата/время регистрации,Крайний срок обработки,Фактическое время выполнения,Плановое время выполнения,Приоритет,Просрочен,Результат работ,Получатель услуг,Инициатор,Кем решен (сотрудник),Расположение,Описание,Услуга,Вид запроса
0,2020-07-29 16:57:41.315,2020-08-17 11:27:50.526,2020-07-30 08:56:31.214,2020-08-10 11:27:41.315,(4) Низкий,не просрочен,Решение предоставлено,User_5690b13b,User_5690b13b,User_27e54663,АЭС-1 (Северная),Добрый день! синий экран при запуске. Ошибка: ...,Рабочая станция (стандарт),4. Настройка / обновление ПО
1,2020-09-03 17:12:25.514,2020-09-22 11:48:43.947,2020-09-04 10:37:38.608,2020-09-15 11:00:00.000,(4) Низкий,не просрочен,Решение предоставлено,User_9d645efd,User_9d645efd,User_c79cd486,АЭС-2 (Западная),Добрый день! прошу установить СЭД на компьютер...,Рабочая станция (стандарт),Запрос на обслуживание
2,2020-12-24 11:10:54.874,2021-01-15 15:28:52.818,2020-12-24 16:23:14.968,2020-12-31 15:25:54.874,(4) Низкий,не просрочен,Решение предоставлено,User_7b6353d9,User_2e984f36,User_2e984f36,АЭС-3 (Восточная),Добрый день! зависает 1С:Предприятие при работ...,Рабочая станция (стандарт),Настройка / обновление ПО
3,2021-02-12 15:05:12.186,2021-03-05 09:37:28.359,2021-02-12 15:23:53.645,2021-02-26 09:35:12.186,(4) Низкий,не просрочен,Решение предоставлено,User_209ccc4f,User_209ccc4f,User_27e54663,АЭС-1 (Северная),Добрый день! после обновления СЭД перестал раб...,Рабочая станция (стандарт),4. Настройка / обновление ПО
4,2021-02-11 08:40:34.669,2021-02-25 12:59:19.790,2021-02-15 11:15:40.259,2021-02-16 15:15:00.000,(4) Низкий,не просрочен,Отменен,User_2f226a4b,User_2f226a4b,User_7881f410,АЭС-4 (Южная),На рабочей станции не включается. Требуется по...,Рабочая станция (КЦ),Запрос на обслуживание


In [17]:
# import pandas as pd
# import numpy as np

# # ========== ЗАГРУЗКА ==========
# df = pd.read_csv(r'C:\Users\Incognitus\Documents\git_hub\Simulate\for_resume_project\support_tickets_final_anonymized.csv',
#                  parse_dates=['Дата/время регистрации', 'Фактическое время выполнения'])

# print(f"Всего строк в исходном файле: {len(df)}")

# # ========== ФИЛЬТР ПО ПОСЛЕДНИМ 6 МЕСЯЦАМ ==========
# # Определяем максимальную дату в данных
# max_date = df['Дата/время регистрации'].max()
# print(f"Максимальная дата в данных: {max_date.date()}")

# # Вычисляем дату 6 месяцев назад от максимальной
# six_months_ago = max_date - pd.DateOffset(months=6)

# # Фильтруем данные за последние 6 месяцев
# df = df[df['Дата/время регистрации'] >= six_months_ago]

# print(f"Период анализа: с {six_months_ago.date()} по {max_date.date()}")
# print(f"Строк за последние 6 месяцев: {len(df)}")

# # Удаляем пустые описания
# df = df.dropna(subset=['Описание'])
# print(f"После удаления пустых описаний: {len(df)}")

# # ========== РАСШИРЕННАЯ ФУНКЦИЯ ИЗВЛЕЧЕНИЯ ПРОБЛЕМ ==========
# def extract_problem_advanced(description):
#     desc = str(description).lower()
    
#     # Приоритет 1: Конкретные ошибки и продукты
#     if 'синий экран' in desc or '0x80070570' in desc or 'bsod' in desc:
#         return 'Синий экран / ошибка Windows'
    
#     if '1с' in desc and ('зависа' in desc or 'тормоз' in desc or 'ошибк' in desc):
#         return 'Зависание/ошибки 1С'
    
#     if 'сэд' in desc:
#         if 'сохранен' in desc or 'сохранить' in desc:
#             return 'СЭД: проблемы с сохранением'
#         elif 'открыть' in desc or 'открывается' in desc:
#             return 'СЭД: проблемы с открытием'
#         elif 'обновлени' in desc:
#             return 'СЭД: проблемы после обновления'
#         else:
#             return 'СЭД: прочие проблемы'
    
#     if 'принтер' in desc or 'печать' in desc or 'мфу' in desc:
#         if 'не печата' in desc:
#             return 'Принтер: не печатает'
#         elif 'картридж' in desc:
#             return 'Принтер: замена картриджа'
#         else:
#             return 'Принтер: прочие проблемы'
    
#     if 'почт' in desc or 'outlook' in desc:
#         if 'не отправля' in desc or 'не приходят' in desc:
#             return 'Почта: проблемы с отправкой/получением'
#         elif 'не открыва' in desc:
#             return 'Почта: не открывается'
#         else:
#             return 'Почта: прочие проблемы'
    
#     if 'доступ' in desc or 'права' in desc or 'ad' in desc or 'active directory' in desc:
#         return 'Проблемы с доступом/правами'
    
#     if 'интернет' in desc or 'сеть' in desc or 'не работает интернет' in desc:
#         return 'Проблемы с сетью/интернетом'
    
#     if 'не включается' in desc or 'не запускается' in desc:
#         return 'Не включается ПК'
    
#     if 'обновлени' in desc and 'после' in desc:
#         return 'Проблемы после обновления'
    
#     if 'установк' in desc or 'установить' in desc:
#         return 'Установка ПО'
    
#     if 'зависа' in desc or 'тормоз' in desc:
#         return 'Зависание/тормозит ПК'
    
#     if 'медленно' in desc or 'долго' in desc:
#         return 'Медленная работа ПК'
    
#     if 'вирус' in desc or 'антивирус' in desc:
#         return 'Проблемы с антивирусом'
    
#     if 'сбой' in desc or 'ошибк' in desc:
#         return 'Сбои/ошибки (неуточненные)'
    
#     return 'Другое'

# # Применяем функцию
# df['ключевая_проблема'] = df['Описание'].apply(extract_problem_advanced)

# # ========== ДОБАВЛЯЕМ ВСПОМОГАТЕЛЬНЫЕ ПОЛЯ ==========
# df['месяц_год'] = df['Дата/время регистрации'].dt.to_period('M').astype(str)
# df['день_недели'] = df['Дата/время регистрации'].dt.day_name()
# df['время_решения_часы'] = (df['Фактическое время выполнения'] - df['Дата/время регистрации']).dt.total_seconds() / 3600
# df['is_critical'] = (df['Приоритет'] == '(1) Критический').astype(int)

# # ========== СОЗДАЕМ ДЕТАЛЬНУЮ ТАБЛИЦУ ==========
# df_powerbi = df[[
#     'Дата/время регистрации',
#     'месяц_год',
#     'день_недели',
#     'Расположение',
#     'Услуга',
#     'Вид запроса',
#     'Приоритет',
#     'ключевая_проблема',
#     'время_решения_часы',
#     'is_critical',
#     'Просрочен',
#     'Результат работ'
# ]].copy()

# df_powerbi.columns = [
#     'Дата регистрации',
#     'Месяц_год',
#     'День недели',
#     'Расположение',
#     'Услуга',
#     'Вид запроса',
#     'Приоритет',
#     'Ключевая проблема',
#     'Время решения (часы)',
#     'Критический',
#     'Просрочен',
#     'Результат'
# ]

# # ========== СОЗДАЕМ АГРЕГИРОВАННУЮ ТАБЛИЦУ ==========
# df_aggregated = df_powerbi.groupby([
#     'Расположение',
#     'Услуга',
#     'Вид запроса',
#     'Приоритет',
#     'Ключевая проблема'
# ]).agg({
#     'Дата регистрации': 'count',
#     'Время решения (часы)': 'mean',
#     'Критический': 'sum',
#     'Просрочен': lambda x: (x == 'просрочен').sum()
# }).reset_index()

# df_aggregated.columns = [
#     'Расположение',
#     'Услуга',
#     'Вид запроса',
#     'Приоритет',
#     'Ключевая проблема',
#     'Количество обращений',
#     'Среднее время решения (часы)',
#     'Критические обращения',
#     'Просроченные обращения'
# ]

# # ========== ТОП-5 ПРОБЛЕМ ==========
# top5_problems = df_powerbi['Ключевая проблема'].value_counts().reset_index()
# top5_problems.columns = ['Проблема', 'Количество']
# top5_problems['Доля %'] = (top5_problems['Количество'] / top5_problems['Количество'].sum() * 100).round(1)

# # ========== ABC-АНАЛИЗ ==========
# problem_stats = df_powerbi.groupby('Ключевая проблема').size().reset_index(name='Количество')
# problem_stats = problem_stats.sort_values('Количество', ascending=False)
# problem_stats['Накопленный процент'] = (problem_stats['Количество'].cumsum() / problem_stats['Количество'].sum() * 100).round(1)

# def get_abc_class(pct):
#     if pct <= 80:
#         return 'A'
#     elif pct <= 95:
#         return 'B'
#     else:
#         return 'C'

# problem_stats['ABC класс'] = problem_stats['Накопленный процент'].apply(get_abc_class)

# # ========== СОХРАНЯЕМ ==========
# df_powerbi.to_csv('data_detailed.csv', index=False, encoding='utf-8-sig')
# df_aggregated.to_csv('data_aggregated.csv', index=False, encoding='utf-8-sig')
# top5_problems.to_csv('top5_problems.csv', index=False, encoding='utf-8-sig')
# problem_stats.to_csv('problem_abc_analysis.csv', index=False, encoding='utf-8-sig')

# # ========== ВЫВОДИМ РЕЗУЛЬТАТЫ ==========
# print("\n" + "="*60)
# print("РЕЗУЛЬТАТЫ АНАЛИЗА ЗА ПОСЛЕДНИЕ 6 МЕСЯЦЕВ")
# print("="*60)

# print("\nТОП-10 ПРИЧИН СБОЕВ:")
# top10 = top5_problems.head(10)
# for i, row in top10.iterrows():
#     print(f"{i+1}. {row['Проблема']}: {row['Количество']} ({row['Доля %']}%)")

# print(f"\n📊 Доля 'Другое': {top5_problems[top5_problems['Проблема'] == 'Другое']['Доля %'].values[0] if len(top5_problems[top5_problems['Проблема'] == 'Другое']) > 0 else 0}%")

# print("\n🏆 ТОП-5 КОНКРЕТНЫХ ПРОБЛЕМ (без 'Другое'):")
# top5_real = top5_problems[top5_problems['Проблема'] != 'Другое'].head(5)
# for i, row in top5_real.iterrows():
#     print(f"{i+1}. {row['Проблема']}: {row['Количество']} ({row['Доля %']}%)")

# print("\n✅ Файлы готовы для Power BI (данные только за последние 6 месяцев)")


# import pandas as pd
# import numpy as np

# # Загрузка данных
# # df = pd.read_csv('../data/raw/support_tickets_final_anonymized.csv',
# #                  parse_dates=['Дата/время регистрации', 'Фактическое время выполнения'])

# # Фильтр за последние 6 месяцев
# max_date = df['Дата/время регистрации'].max()
# six_months_ago = max_date - pd.DateOffset(months=6)
# df = df[df['Дата/время регистрации'] >= six_months_ago]
# df = df.dropna(subset=['Описание'])

# # Функция извлечения ключевой проблемы
# def extract_problem(desc):
#     desc = str(desc).lower()
    
#     if 'синий экран' in desc or '0x80070570' in desc:
#         return 'Синий экран / ошибка Windows'
#     if '1с' in desc and ('зависа' in desc or 'тормоз' in desc):
#         return 'Зависание/ошибки 1С'
#     if 'сэд' in desc:
#         if 'сохранен' in desc:
#             return 'СЭД: проблемы с сохранением'
#         if 'открыть' in desc:
#             return 'СЭД: проблемы с открытием'
#         return 'СЭД: прочие проблемы'
#     if 'принтер' in desc or 'печать' in desc:
#         return 'Проблемы с печатью'
#     if 'почт' in desc or 'outlook' in desc:
#         return 'Проблемы с почтой'
#     if 'доступ' in desc or 'права' in desc:
#         return 'Проблемы с доступом/правами'
#     if 'интернет' in desc or 'сеть' in desc:
#         return 'Проблемы с сетью'
#     if 'не включается' in desc:
#         return 'Не включается ПК'
#     if 'обновлени' in desc:
#         return 'Проблемы после обновления'
#     if 'установк' in desc:
#         return 'Установка ПО'
#     if 'зависа' in desc or 'тормоз' in desc:
#         return 'Зависание/тормозит ПК'
#     if 'сбой' in desc or 'ошибк' in desc:
#         return 'Сбои/ошибки (неуточненные)'
    
#     return 'Другое'

# df['Ключевая проблема'] = df['Описание'].apply(extract_problem)

# # Добавление вспомогательных полей
# df['Месяц_год'] = df['Дата/время регистрации'].dt.to_period('M').astype(str)
# df['День недели'] = df['Дата/время регистрации'].dt.day_name()
# df['Время решения (часы)'] = (df['Фактическое время выполнения'] - df['Дата/время регистрации']).dt.total_seconds() / 3600
# df['Критический'] = (df['Приоритет'] == '(1) Критический').astype(int)

# # Детальная таблица для Power BI
# df_detailed = df[[
#     'Дата/время регистрации', 'Месяц_год', 'День недели',
#     'Расположение', 'Услуга', 'Вид запроса', 'Приоритет',
#     'Ключевая проблема', 'Время решения (часы)', 'Критический',
#     'Просрочен', 'Результат работ'
# ]].copy()

# df_detailed.columns = [
#     'Дата регистрации', 'Месяц_год', 'День недели',
#     'Расположение', 'Услуга', 'Вид запроса', 'Приоритет',
#     'Ключевая проблема', 'Время решения (часы)', 'Критический',
#     'Просрочен', 'Результат'
# ]

# # Топ-5 проблем
# top5 = df_detailed['Ключевая проблема'].value_counts().reset_index()
# top5.columns = ['Проблема', 'Количество']
# top5['Доля %'] = (top5['Количество'] / top5['Количество'].sum() * 100).round(1)

# # ABC-анализ
# abc = df_detailed.groupby('Ключевая проблема').size().reset_index(name='Количество')
# abc = abc.sort_values('Количество', ascending=False)
# abc['Накопленный процент'] = (abc['Количество'].cumsum() / abc['Количество'].sum() * 100).round(1)
# abc['ABC класс'] = abc['Накопленный процент'].apply(lambda x: 'A' if x <= 80 else ('B' if x <= 95 else 'C'))

# # Сохранение
# # df_detailed.to_csv('../data/processed/data_detailed.csv', index=False, encoding='utf-8-sig')
# # top5.to_csv('../data/processed/top5_problems.csv', index=False, encoding='utf-8-sig')
# # abc.to_csv('../data/processed/problem_abc_analysis.csv', index=False, encoding='utf-8-sig')

# df_detailed.to_csv('data_detailed.csv', index=False, encoding='utf-8-sig')
# top5.to_csv('top5_problems.csv', index=False, encoding='utf-8-sig')
# abc.to_csv('problem_abc_analysis.csv', index=False, encoding='utf-8-sig')

# print("✅ Готово. Файлы сохранены в data/processed/")

import pandas as pd
import numpy as np

# Загрузка данных
# df = pd.read_csv('../data/raw/support_tickets_final_anonymized.csv',
#                  parse_dates=['Дата/время регистрации', 'Фактическое время выполнения'])

print("="*60)
print("СТАТИСТИКА ПО ДАННЫМ")
print("="*60)

print(f"\nВсего строк в исходном файле: {len(df)}")

# Фильтр за последние 6 месяцев
max_date = df['Дата/время регистрации'].max()
six_months_ago = max_date - pd.DateOffset(months=6)
df = df[df['Дата/время регистрации'] >= six_months_ago]

print(f"Максимальная дата в данных: {max_date.date()}")
print(f"Период анализа: с {six_months_ago.date()} по {max_date.date()}")
print(f"Строк за последние 6 месяцев: {len(df)}")

df = df.dropna(subset=['Описание'])
print(f"После удаления пустых описаний: {len(df)}")

# Функция извлечения ключевой проблемы
def extract_problem(desc):
    desc = str(desc).lower()
    
    if 'синий экран' in desc or '0x80070570' in desc:
        return 'Синий экран / ошибка Windows'
    if '1с' in desc and ('зависа' in desc or 'тормоз' in desc):
        return 'Зависание/ошибки 1С'
    if 'сэд' in desc:
        if 'сохранен' in desc:
            return 'СЭД: проблемы с сохранением'
        if 'открыть' in desc:
            return 'СЭД: проблемы с открытием'
        return 'СЭД: прочие проблемы'
    if 'принтер' in desc or 'печать' in desc:
        return 'Проблемы с печатью'
    if 'почт' in desc or 'outlook' in desc:
        return 'Проблемы с почтой'
    if 'доступ' in desc or 'права' in desc:
        return 'Проблемы с доступом/правами'
    if 'интернет' in desc or 'сеть' in desc:
        return 'Проблемы с сетью'
    if 'не включается' in desc:
        return 'Не включается ПК'
    if 'обновлени' in desc:
        return 'Проблемы после обновления'
    if 'установк' in desc:
        return 'Установка ПО'
    if 'зависа' in desc or 'тормоз' in desc:
        return 'Зависание/тормозит ПК'
    if 'сбой' in desc or 'ошибк' in desc:
        return 'Сбои/ошибки (неуточненные)'
    
    return 'Другое'

df['Ключевая проблема'] = df['Описание'].apply(extract_problem)

# Добавление вспомогательных полей
df['Месяц_год'] = df['Дата/время регистрации'].dt.to_period('M').astype(str)
df['День недели'] = df['Дата/время регистрации'].dt.day_name()
df['Время решения (часы)'] = (df['Фактическое время выполнения'] - df['Дата/время регистрации']).dt.total_seconds() / 3600
df['Критический'] = (df['Приоритет'] == '(1) Критический').astype(int)

# Детальная таблица для Power BI
df_detailed = df[[
    'Дата/время регистрации', 'Месяц_год', 'День недели',
    'Расположение', 'Услуга', 'Вид запроса', 'Приоритет',
    'Ключевая проблема', 'Время решения (часы)', 'Критический',
    'Просрочен', 'Результат работ'
]].copy()

df_detailed.columns = [
    'Дата регистрации', 'Месяц_год', 'День недели',
    'Расположение', 'Услуга', 'Вид запроса', 'Приоритет',
    'Ключевая проблема', 'Время решения (часы)', 'Критический',
    'Просрочен', 'Результат'
]

# Топ-5 проблем
top5 = df_detailed['Ключевая проблема'].value_counts().reset_index()
top5.columns = ['Проблема', 'Количество']
top5['Доля %'] = (top5['Количество'] / top5['Количество'].sum() * 100).round(1)

# ABC-анализ
abc = df_detailed.groupby('Ключевая проблема').size().reset_index(name='Количество')
abc = abc.sort_values('Количество', ascending=False)
total = abc['Количество'].sum()
abc['Накопленный процент'] = (abc['Количество'].cumsum() / total * 100).round(1)
abc['ABC класс'] = abc['Накопленный процент'].apply(lambda x: 'A' if x <= 80 else ('B' if x <= 95 else 'C'))

# Сохранение
df_detailed.to_csv('data_detailed.csv', index=False, encoding='utf-8-sig')
top5.to_csv('top5_problems.csv', index=False, encoding='utf-8-sig')
abc.to_csv('problem_abc_analysis.csv', index=False, encoding='utf-8-sig')

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ АНАЛИЗА ЗА ПОСЛЕДНИЕ 6 МЕСЯЦЕВ")
print("="*60)

print("\nТОП-10 ПРИЧИН СБОЕВ:")
for i, row in top5.head(10).iterrows():
    print(f"{i+1}. {row['Проблема']}: {row['Количество']} ({row['Доля %']}%)")

print(f"\n📊 Доля 'Другое': {top5[top5['Проблема'] == 'Другое']['Доля %'].values[0] if len(top5[top5['Проблема'] == 'Другое']) > 0 else 0}%")

print("\n🏆 ТОП-5 КОНКРЕТНЫХ ПРОБЛЕМ (без 'Другое'):")
top5_real = top5[top5['Проблема'] != 'Другое'].head(5)
for i, row in top5_real.iterrows():
    print(f"{i+1}. {row['Проблема']}: {row['Количество']} ({row['Доля %']}%)")

print(f"\n📋 ABC-анализ (первые 10):")
for i, row in abc.head(10).iterrows():
    print(f"   {row['Ключевая проблема']}: {row['Количество']} ({row['Накопленный процент']}%) - {row['ABC класс']}")

print(f"\n✅ Всего обработано: {len(df_detailed)} обращений")
print("✅ Файлы сохранены: data_detailed.csv, top5_problems.csv, problem_abc_analysis.csv")

СТАТИСТИКА ПО ДАННЫМ

Всего строк в исходном файле: 18992
Максимальная дата в данных: 2025-11-18
Период анализа: с 2025-05-18 по 2025-11-18
Строк за последние 6 месяцев: 7261
После удаления пустых описаний: 7261

РЕЗУЛЬТАТЫ АНАЛИЗА ЗА ПОСЛЕДНИЕ 6 МЕСЯЦЕВ

ТОП-10 ПРИЧИН СБОЕВ:
1. Другое: 2569 (35.4%)
2. Синий экран / ошибка Windows: 1168 (16.1%)
3. Зависание/тормозит ПК: 1097 (15.1%)
4. СЭД: прочие проблемы: 649 (8.9%)
5. Проблемы после обновления: 523 (7.2%)
6. Сбои/ошибки (неуточненные): 289 (4.0%)
7. Зависание/ошибки 1С: 233 (3.2%)
8. Не включается ПК: 205 (2.8%)
9. Проблемы с почтой: 199 (2.7%)
10. Проблемы с печатью: 179 (2.5%)

📊 Доля 'Другое': 35.4%

🏆 ТОП-5 КОНКРЕТНЫХ ПРОБЛЕМ (без 'Другое'):
2. Синий экран / ошибка Windows: 1168 (16.1%)
3. Зависание/тормозит ПК: 1097 (15.1%)
4. СЭД: прочие проблемы: 649 (8.9%)
5. Проблемы после обновления: 523 (7.2%)
6. Сбои/ошибки (неуточненные): 289 (4.0%)

📋 ABC-анализ (первые 10):
   Другое: 2569 (35.4%) - A
   Синий экран / ошибка Windows: 